<a href="https://colab.research.google.com/github/phucsz/DAAI_N1.4/blob/main/Problem_statement_1%2B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem statement 1:** Ánh xạ và tổng hợp dữ liệu doanh thu chi tiết phân cấp theotừng Thành phố (City) và vùng miền (Region), quận (district) để khoanh vùng các thị
trường phát triển mạnh nhất.

In [8]:
# ==========================================
# BƯỚC 1: IMPORT THƯ VIỆN & ĐỌC DỮ LIỆU
# ==========================================
import pandas as pd
import plotly.express as px

# Xóa dấu # ở đầu dòng để Python thực sự đọc file
# Hãy đảm bảo tên file trong ngoặc kép khớp 100% với tên file bạn vừa tải lên ở mép trái Colab
order_enriched = pd.read_csv('orders_enriched.csv')
order_items = pd.read_csv('order_items.csv')

# Kiểm tra xem dữ liệu đã đọc thành công chưa
print("Đã load xong dữ liệu!")

Đã load xong dữ liệu!


/tmp/ipykernel_2003/1115087584.py:10: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('order_items.csv')


In [9]:
# ==========================================
# BƯỚC 1: IMPORT THƯ VIỆN & ĐỌC DỮ LIỆU
# ==========================================
import pandas as pd
import plotly.express as px

# Giả sử bạn tải file lên Colab, dùng code đọc file CSV (nhớ sửa lại tên file cho đúng)
# order_enriched = pd.read_csv('order_enriched.csv')
# order_items = pd.read_csv('order_items.csv')

# ==========================================
# BƯỚC 2: TẠO "MAJOR" TÍNH DOANH THU THỰC TẾ
# ==========================================
# Xử lý các giá trị rỗng (NaN) trong cột discount_amount thành 0 để không bị lỗi khi trừ
order_items['discount_amount'] = order_items['discount_amount'].fillna(0)

# Tính Doanh thu (Revenue) cho từng mặt hàng: (Số lượng * Đơn giá) - Giảm giá
order_items['Revenue'] = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount']

# ==========================================
# BƯỚC 3: KẾT NỐI BẢNG (MERGE)
# ==========================================
# Lấy cột order_id và các cột địa lý từ bảng order_enriched
geo_columns = order_enriched[['order_id', 'region', 'city', 'district']]

# Ghép dữ liệu Doanh thu (order_items) với dữ liệu Địa lý (geo_columns) thông qua order_id
df_final = pd.merge(order_items, geo_columns, on='order_id', how='left')

# ==========================================
# BƯỚC 4: TỔNG HỢP & KHOANH VÙNG THỊ TRƯỜNG
# ==========================================
# Gom nhóm dữ liệu theo phân cấp (Region -> City -> District) và tính tổng Doanh thu
summary_df = df_final.groupby(['region', 'city', 'district'])['Revenue'].sum().reset_index()

# Sắp xếp doanh thu từ cao xuống thấp
summary_df = summary_df.sort_values(by='Revenue', ascending=False)

# In ra màn hình Bảng Top 10 thị trường mạnh nhất để đưa vào báo cáo
print("TOP 10 THỊ TRƯỜNG CÓ DOANH THU CAO NHẤT:")
display(summary_df.head(10))

# ==========================================
# BƯỚC 5: ÁNH XẠ TRỰC QUAN BẰNG TREEMAP
# ==========================================
# Vẽ biểu đồ phân cấp (Hierarchy) từ Vùng miền xuống tận Quận huyện
fig = px.treemap(summary_df,
                 path=[px.Constant("Tổng Doanh Thu"), 'region', 'city', 'district'],
                 values='Revenue',
                 color='Revenue',
                 color_continuous_scale='Blues',
                 title='Ánh xạ Doanh thu theo Địa giới (Region -> City -> District)')

# Hiển thị số liệu trực tiếp trên biểu đồ
fig.update_traces(textinfo="label+value")
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))

# Show biểu đồ tương tác
fig.show()

TOP 10 THỊ TRƯỜNG CÓ DOANH THU CAO NHẤT:


,region,city,district,Revenue
374,East,Thai Nguyen,District #10,72569346.33
497,West,Pleiku,District #38,71667351.96
531,West,Vung Tau,District #37,68022800.14
54,Central,Kon Tum,District #22,65232581.46
16,Central,Dong Hoi,District #23,60887299.20
120,Central,Quy Nhon,District #23,59346852.54
42,Central,Hue,District #23,58572037.44
510,West,Soc Trang,District #37,56086358.48
476,West,Ho Chi Minh City,District #38,55661182.62
102,Central,Phan Thiet,District #31,55614447.17


**Problem statement 4:** Phân tích Cơ cấu Doanh thu và Thị phần theo Khu vực Địa lý

In [10]:
import pandas as pd
import plotly.express as px

# ==========================================
# BƯỚC 1: CHUẨN BỊ DỮ LIỆU & TÍNH DOANH THU
# ==========================================
order_items['discount_amount'] = order_items['discount_amount'].fillna(0)
order_items['Revenue'] = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount']

# Lấy các cột cần thiết, đặc biệt là phải có order_date
cols_to_merge = order_enriched[['order_id', 'order_date', 'region', 'city', 'district']]
df = pd.merge(order_items, cols_to_merge, on='order_id', how='left')

# Chuyển cột order_date về định dạng thời gian chuẩn của Python
df['order_date'] = pd.to_datetime(df['order_date'])

# ==========================================
# BƯỚC 2: TÍNH CƠ CẤU THỊ PHẦN (MARKET SHARE)
# ==========================================
total_revenue = df['Revenue'].sum()

# Tính tổng doanh thu theo từng Region
region_df = df.groupby('region')['Revenue'].sum().reset_index()

# Tính % Tỷ trọng
region_df['Market_Share_%'] = (region_df['Revenue'] / total_revenue) * 100

# Vẽ Biểu đồ tròn (Donut Chart) thể hiện Cơ cấu Doanh thu
fig_pie = px.pie(region_df, values='Revenue', names='region', hole=0.4,
                 title='Cơ cấu Doanh thu theo Khu vực (Region)')
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.show()

# ==========================================
# BƯỚC 3: TÍNH TỐC ĐỘ TĂNG TRƯỞNG & LẬP MA TRẬN
# ==========================================
# Mẹo tính tăng trưởng: Ta chia đôi khoảng thời gian dữ liệu thành 2 nửa:
# "Quá khứ" (Nửa đầu) và "Gần đây" (Nửa sau) để xem doanh thu có tăng lên không.
median_date = df['order_date'].median()
df['Period'] = df['order_date'].apply(lambda x: 'Gần đây' if x > median_date else 'Quá khứ')

# Tính tổng tiền của 2 giai đoạn cho từng Region
growth_df = df.groupby(['region', 'Period'])['Revenue'].sum().unstack().reset_index()

# Công thức % Tăng trưởng = (Gần đây - Quá khứ) / Quá khứ
growth_df['Growth_Rate_%'] = ((growth_df['Gần đây'] - growth_df['Quá khứ']) / growth_df['Quá khứ']) * 100

# Ghép Thị phần và Tăng trưởng lại thành 1 bảng báo cáo cuối cùng
ps4_report = pd.merge(region_df, growth_df[['region', 'Growth_Rate_%']], on='region')

print("BẢNG ĐÁNH GIÁ THỊ PHẦN VÀ TĂNG TRƯỞNG:")
display(ps4_report.round(2).sort_values(by='Revenue', ascending=False))

# ==========================================
# BƯỚC 4: VẼ MA TRẬN CHIẾN LƯỢC (BCG MATRIX)
# ==========================================
# Đây là biểu đồ quan trọng nhất để trả lời câu hỏi của ban giám đốc
fig_bcg = px.scatter(ps4_report,
                     x='Market_Share_%',
                     y='Growth_Rate_%',
                     size='Revenue', # Kích thước bong bóng là Doanh thu
                     color='region',
                     text='region',
                     title='Ma trận Đánh giá Chiến lược Khu vực (Thị phần vs Tốc độ Tăng trưởng)')

fig_bcg.update_traces(textposition='top center', marker=dict(line=dict(width=1, color='DarkSlateGrey')))

# Vẽ 2 đường gạch ngang/dọc chia biểu đồ làm 4 góc phần tư
fig_bcg.add_hline(y=ps4_report['Growth_Rate_%'].mean(), line_dash="dash", line_color="red", annotation_text="Tăng trưởng trung bình")
fig_bcg.add_vline(x=ps4_report['Market_Share_%'].mean(), line_dash="dash", line_color="blue", annotation_text="Thị phần trung bình")
fig_bcg.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig_bcg.show()

BẢNG ĐÁNH GIÁ THỊ PHẦN VÀ TĂNG TRƯỞNG:


,region,Revenue,Market_Share_%,Growth_Rate_%
1,East,7.291151e+09,46.50,19.50
0,Central,4.719491e+09,30.10,33.86
2,West,3.670227e+09,23.41,13.17
